# Move public stirrup/tie images into our Roboflow project

Copies images (no labels) from two public Universe datasets into **Concrete Evidence column cages**, where we outline them as `bar · tie · hook_90 · hook_135`.

Runs on CPU — no GPU needed. Run all cells; paste your Roboflow API key when asked (Roboflow → Settings → API Keys). The key is not saved in the notebook.

In [ ]:
!pip install -q roboflow

In [ ]:
from getpass import getpass
from roboflow import Roboflow
import glob, os
rf = Roboflow(api_key=getpass('Roboflow API key: '))
dest = rf.workspace('akshat-dayal-students-iaac-net').project('concrete-evidence-column-cages')
print('target:', dest.name)

In [ ]:
# (workspace, project, version, tag) - public Universe datasets, CC BY 4.0
SRC = [('column-tie-detection', 'custom-workflow-object-detection-yhfbh', 7, 'column_tie'),
       ('steelbinding-for-rcc', 'steelbindings', 1, 'steelbindings')]
# stirrup-recognition-dataset is skipped: it has no published version and generating one costs credits.

for ws, proj, ver, tag in SRC:
    loc = f'/content/rf_src/{tag}'
    rf.workspace(ws).project(proj).version(ver).download('yolov8', location=loc)
    imgs = sorted(glob.glob(f'{loc}/*/images/*'))
    print(f'{tag}: {len(imgs)} images -> uploading')
    ok = 0
    for i, p in enumerate(imgs):
        try:
            dest.upload(image_path=p, batch_name=tag, tag_names=[tag], num_retry_uploads=2)
            ok += 1
        except Exception as e:
            print('  failed', os.path.basename(p), e)
        if i % 50 == 0:
            print(f'  {i}/{len(imgs)}')
    print(f'{tag}: uploaded {ok}/{len(imgs)}')
print('All done - open Roboflow > Annotate to start outlining.')